# The comparator at the second number

`max(n1, n2, n3)` on Qwen2.5-7B-Instruct, one-shot prompt. Everything is read at the **second**
number position $n_2$. Attention is causal, so only $y_1$ and $y_2$ exist there:

$$\mathbf{w}_1 = \text{L13 DAS 1D @ } n_2 \ (y_1),\qquad
  \mathbf{w}_2 = \text{L13 DAS 1D @ } n_2 \ (y_2)$$

Two perturbation cases, the same orders `explore_multicompare` uses: `y1` ($s_1>s_3>s_2$, the
maximum at $n_1$) and `y2` ($s_2>s_3>s_1$, the maximum at $n_2$). No `y3` case — its
counterfactual changes a token to the right of the patch site.

Neurons are **not** re-derived here: the 12 the two-number comparator identified are carried over
and measured at $n_2$. Every block caches its `.npz` and draws its figure immediately. See
`docs/explore_multicompare2.md`.

In [ ]:
import os, time, warnings, importlib
import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt

import data_numberreps as D
import data_sharedrep as S
import data_comparator as C
import data_multicompare as MC
import data_multicompare2 as M2
importlib.reload(MC); importlib.reload(M2)
from data_multicompare import (LAYER, H_LAYER, REGIMES, N_EVAL, N_TRAIN, N_CLOUD, ck)
from data_multicompare2 import CASES2 as CASES, CASE_LIST2 as CASE_LIST
from data_numberreps import SEED, EVAL_BS, device

RESULTS_DIR, FIGS_DIR = 'results', 'figs'
os.makedirs(FIGS_DIR, exist_ok=True)

REGIME = '2digit'
LOW, HIGH, GAP = (REGIMES[REGIME][k] for k in ('low', 'high', 'gap'))

NEURON_LAYERS = [14, 15]
D_FF = 18944                    # Qwen2.5-7B intermediate size; asserted against the model below
POOL_LAYER  = np.concatenate([np.full(D_FF, l) for l in NEURON_LAYERS])
POOL_WITHIN = np.concatenate([np.arange(D_FF) for _ in NEURON_LAYERS])
NAME = lambda p: f'L{POOL_LAYER[p]}#{POOL_WITHIN[p]}'

# The n3 notebook's directions, for the cross-position comparisons in sections 1 and 13, and the
# two-number neurons for section 11. Left as lazy NpzFiles: `dirs` holds 80 MB of clouds.
N3 = {b: (np.load(f'{RESULTS_DIR}/multicompare_{b}_{REGIME}.npz')
          if os.path.exists(f'{RESULTS_DIR}/multicompare_{b}_{REGIME}.npz') else None)
      for b in ('dirs', 'ucos')}
TWO_PATH = f'{RESULTS_DIR}/comparator_data_{REGIME}.npz'
TWO = np.load(TWO_PATH) if os.path.exists(TWO_PATH) else None
print('n3 results:', {b: ('loaded' if v is not None else 'MISSING') for b, v in N3.items()})
print('two-number results:', 'loaded' if TWO is not None else f'MISSING ({TWO_PATH})')

## Plot style

In [ ]:
LABEL_FS, TICK_FS, LEGEND_FS, TITLE_FS = 16, 13, 13, 17
DPI, SAVE = 300, True

# Same palette as explore_multicompare.
COL = {'full': '#2166AC',   # blue
       'das':  '#C97B00',   # amber
       'pc1':  '#C06A9B',   # orchid
       'h14':  '#117777',   # teal
       'ceil': '#6E6E6E'}   # grey

NCOL = {1: '#C97B00', 2: '#2166AC', 3: '#2E8B57'}
VCOL = {1: '#117777', 2: '#C06A9B', 3: '#6A51A3'}
CASE_COL = {'y1': NCOL[1], 'y2': NCOL[2]}
JOINT_HATCH = '//'

W = {1: r'$\mathbf{w}_1$', 2: r'$\mathbf{w}_2$'}
Y = {1: r'$y_1$', 2: r'$y_2$', 3: r'$y_3$'}
CMAP = 'viridis'


def style_axes(ax, xlabel=None, ylabel=None, title=None, grid=True):
    if xlabel: ax.set_xlabel(xlabel, fontsize=LABEL_FS)
    if ylabel: ax.set_ylabel(ylabel, fontsize=LABEL_FS)
    if title:  ax.set_title(title, fontsize=TITLE_FS)
    ax.tick_params(labelsize=TICK_FS)
    if grid: ax.grid(alpha=0.3, axis='y')
    for side in ('top', 'right'):
        if side in ax.spines: ax.spines[side].set_visible(False)


def savefig(fig, name):
    if SAVE:
        fig.savefig(f'{FIGS_DIR}/multicompare2_{name}.png', dpi=DPI, bbox_inches='tight')


def aggregate(x, y):
    '''Per-distinct-x mean and standard error of y.'''
    u = np.unique(x)
    m = np.array([y[x == v].mean() for v in u])
    se = np.array([y[x == v].std() / max(np.sqrt((x == v).sum()), 1) for v in u])
    return u, m, se


def barlab(ax, xpos, mean, se, text, fs=10):
    ax.text(xpos, mean + se + 0.02, text, ha='center', va='bottom', fontsize=fs)

## Run mode

Each block writes one `.npz`. When all of them exist the model is never loaded and the notebook
runs as a pure plotting pass. The quadruple pool is shared with `explore_multicompare`.

In [ ]:
BLOCKS = ['pool', 'posmap', 'dirs', 'space', 'sweep', 'freeze', 'rf', 'transfer', 'das15']
# section 11 ('ablate') is self-contained and caches itself -- deliberately not in BLOCKS
PATH = {b: f'{RESULTS_DIR}/multicompare2_{b}_{REGIME}.npz' for b in BLOCKS}
# Same seed, same N_POOL, same order filter as explore_multicompare, so the file is identical by
# construction and the two notebooks are measured on the same quadruples.
PATH['pool'] = f'{RESULTS_DIR}/multicompare_pool_{REGIME}.npz'

FORCE = set()        # block names to recompute even though their file exists, e.g. {'sweep'}

_missing = [b for b in BLOCKS if not os.path.exists(PATH[b])]
PLOT_ONLY = not _missing and not FORCE
print('run mode:', 'PLOT ONLY -- every result is cached, the model is not loaded' if PLOT_ONLY
      else f'COMPUTE -- missing: {_missing or "none"}   forced: {sorted(FORCE) or "none"}')


def cached(block, compute):
    '''Load `PATH[block]` if present, else run `compute()` -> {name: array}, save it, return it.'''
    p = PATH[block]
    if os.path.exists(p) and block not in FORCE:
        d = dict(np.load(p, allow_pickle=True))
        print(f'loaded {p}   ({len(d)} arrays)')
        return d
    assert not PLOT_ONLY, f'{p} is not cached -- set PLOT_ONLY = False and run the model cells'
    t = time.time()
    d = compute()
    np.savez_compressed(p, **{k: np.asarray(v) for k, v in d.items()})
    print(f'computed and saved {p}   ({len(d)} arrays, {time.time() - t:.0f} s)')
    return d

## Model and token layout

Separate cell — everything above runs without it. The $y_3$ check is the premise of the whole
notebook: under a causal mask the residual at $n_2$ cannot depend on the third number.

In [ ]:
if PLOT_ONLY:
    model = tokenizer = None
    print('plot-only: model not loaded')
else:
    model, tokenizer = D.load_model()
    S.init_head_machinery()
    assert D.model.config.intermediate_size == D_FF
    assert S.n_heads == D.d_model // D.head_dim

T, POS, NTOK, REF = (None,) * 4
if not PLOT_ONLY:
    T, POS, NTOK, REF = MC.regime_positions(LOW, HIGH)
    print(f'T={T}   number last-tokens {POS}   tokens per number {NTOK}')
    print(f'reference prompt: {REF!r}')
    d = M2.y3_is_unreachable(T, POS[1], LOW, HIGH)
    print('max |delta| at n2 from changing y3: '
          + '  '.join(f'L{l} {v:.3e}' for l, v in d.items()))
    assert max(d.values()) == 0.0, 'y3 must not reach n2'

## The quadruple pool

Loaded from `explore_multicompare`'s cache: the same 400 eval + 128 train quadruples, filtered to
the ones the model solves in every value order.

In [ ]:
N_POOL = 1400            # drawn before filtering; the assert below says if it was not enough
N_KEEP = N_EVAL + N_TRAIN


def _pool():
    quads = MC.build_quads(N_POOL, SEED, LOW, HIGH, GAP)
    ok = np.ones(len(quads), dtype=bool)
    per = {}
    for o in MC.ALL_ORDERS:
        m = MC.solved_mask(MC.make_batch(quads, o, T))
        per[MC.order_label(o)] = m.mean()
        ok &= m
        print(f'  {MC.order_label(o)}: solved {m.mean():.3f}', flush=True)
    idx = np.where(ok)[0]
    assert len(idx) >= N_KEEP, f'only {len(idx)} usable quadruples; raise N_POOL'
    kept = np.array([quads[i] for i in idx[:N_KEEP]], dtype=np.int64)
    return dict(quads=kept, eval_quads=kept[:N_EVAL], train_quads=kept[N_EVAL:],
                solved_frac_all=np.array(len(idx) / len(quads)),
                solved_frac=np.array([per[MC.order_label(o)] for o in MC.ALL_ORDERS]),
                orders=np.array(MC.ALL_ORDERS), T=np.array(T if T else 0),
                pos=np.array(POS if POS else [0, 0, 0]))


POOLD = cached('pool', _pool)
assert len(POOLD['eval_quads']) == N_EVAL and len(POOLD['train_quads']) == N_TRAIN
EVAL_Q = [tuple(q) for q in POOLD['eval_quads']]
TRAIN_Q = [tuple(q) for q in POOLD['train_quads']]
if PLOT_ONLY:
    T, POS = int(POOLD['T']), POOLD['pos'].tolist()
POS1, POS2, POS3 = POS
print(f'{len(EVAL_Q)} eval + {len(TRAIN_Q)} train quadruples   '
      f'tokens n1/n2/n3 = {POS1}/{POS2}/{POS3}   patch site here: n2 = {POS2}')
for c in CASE_LIST:
    o = CASES[c]
    print(f'  case {c}: order {MC.order_label(o)}   max at slot {o[0] + 1}, '
          f'runner-up at slot {o[1] + 1}')

## Batches and clean-run sources

One batch per case, built once and reused by every block. Sources are captured per (case,
position) and cached in the session.

In [ ]:
BAT, HID, CTX, PRE = {}, {}, {}, {}


def get_bat(o):
    if o not in BAT:
        b = MC.make_batch(EVAL_Q, o, T)
        b['pld_clean'], b['pld_corr'] = MC.pos_anchors(b)
        BAT[o] = b
    return BAT[o]


def get_src(o, pos, layers=(LAYER, H_LAYER), need_ctx=True):
    '''The clean-run activations every condition patches from.

    `MC.sources_for` keys everything by (batch, position), but `S.clean_hidden_at` returns the
    whole (B, T, d_model) sequence and is therefore position independent. Section 1 patches at
    three positions, so those captures are cached per (order, layer) and shared; only the o_proj
    context and the pre-MLP residual, which are read at one token, are per position.
    '''
    bat = get_bat(o)
    src = {}
    for l in layers:
        if (o, l) not in HID:
            HID[(o, l)] = S.clean_hidden_at(bat['ids_c'], l)
        src[l] = HID[(o, l)]
    if need_ctx:
        if (o, pos) not in CTX:
            CTX[(o, pos)] = S.capture_ctx(bat['ids_c'], pos)
            PRE[(o, pos)] = S.capture_premlp(bat['ids_c'], pos)
        src['ctx'], src['premlp'] = CTX[(o, pos)], PRE[(o, pos)]
    return src


if not PLOT_ONLY:
    for c in CASE_LIST:
        b = get_bat(CASES[c])
        print(f'{c} ({MC.order_label(CASES[c])}): clean PLD {b["pld_clean"].mean():+6.2f}   '
              f'corrupted PLD {b["pld_corr"].mean():+6.2f}   '
              f'gap {(b["pld_clean"] - b["pld_corr"]).mean():5.2f}', flush=True)

# 1. Where the causal information sits

Full-rank interchange of the L13 and L14 residuals at $n_1$, $n_2$ and $n_3$, same batches
throughout. This is the frame for everything below: which positions carry the perturbed number.

In [ ]:
PNAME = {0: 'n1', 1: 'n2', 2: 'n3'}


def _posmap():
    out = {}
    for case in CASE_LIST:
        o = CASES[case]
        bat = get_bat(o)
        for pi in (0, 1, 2):
            p = POS[pi]
            src = get_src(o, p, need_ctx=False)
            for nm, spec in (('L13', dict(resid=[(LAYER, 'full', None)])),
                             ('L14', dict(resid=[(H_LAYER, 'full', None)]))):
                rec, iia = MC.eval_spec(spec, bat, src, p)
                out[f'posrec_{case}_{PNAME[pi]}_{nm}'] = rec.astype(np.float32)
                out[f'iia_{case}_{PNAME[pi]}_{nm}'] = iia.astype(np.float32)
            print(f'{case} @ {PNAME[pi]}: '
                  + '  '.join(f'{nm} iia {out[f"iia_{case}_{PNAME[pi]}_{nm}"].mean():.3f} '
                              f'pr {out[f"posrec_{case}_{PNAME[pi]}_{nm}"].mean():+.3f}'
                              for nm in ('L13', 'L14')), flush=True)
    return out


POSMAP = cached('posmap', _posmap)
# n1 and n3 are only visited here; everything below patches at n2.
for _k in [k for k in CTX if k[1] != POS2]:
    CTX.pop(_k); PRE.pop(_k)
torch.cuda.empty_cache()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
xs, Wb = np.arange(3), 0.8 / (2 * len(CASE_LIST))
for ax, (key, ylab, ylim) in zip(axes, [('iia', 'IIA', (0, 1.15)),
                                        ('posrec', 'position recovery', None)]):
    for ci, case in enumerate(CASE_LIST):
        for li, nm in enumerate(('L13', 'L14')):
            vals = [POSMAP[f'{key}_{case}_{PNAME[pi]}_{nm}'] for pi in (0, 1, 2)]
            m = [v.mean() for v in vals]
            e = [v.std() / np.sqrt(len(v)) for v in vals]
            ax.bar(xs + (2 * ci + li - 1.5) * Wb, m, Wb, yerr=e, capsize=3,
                   color=CASE_COL[case], alpha=0.92,
                   hatch=JOINT_HATCH if nm == 'L14' else '', edgecolor='white', linewidth=0,
                   label=f'{Y[int(case[1])]} perturbed, {nm} full')
    ax.set_xticks(xs)
    ax.set_xticklabels(['$n_1$', '$n_2$', '$n_3$'], fontsize=LABEL_FS)
    ax.axhline(0, color='0.6', lw=0.9)
    ax.axvline(0.5, color='0.85', lw=0.8); ax.axvline(1.5, color='0.85', lw=0.8)
    if ylim: ax.set_ylim(*ylim)
    style_axes(ax, ylabel=ylab, xlabel='patch position')
    ax.legend(fontsize=LEGEND_FS - 1)
fig.suptitle('full-rank interchange at each number position', fontsize=TITLE_FS)
plt.tight_layout(); savefig(fig, 'position_map'); plt.show()

for case in CASE_LIST:
    print(f'{case}: ' + '   '.join(
        f'{PNAME[pi]} L13 {POSMAP[f"iia_{case}_{PNAME[pi]}_L13"].mean():.3f}'
        f'/{POSMAP[f"posrec_{case}_{PNAME[pi]}_L13"].mean():+.3f}' for pi in (0, 1, 2)))

# 2. The two directions at $n_2$

Both are rank-1 DAS on the L13 residual at $n_2$: $\mathbf{w}_1$ fitted on the `y1` case,
$\mathbf{w}_2$ on `y2`. The whole residual is searched rather than one head's output, so if a
rank-1 direction anywhere carries the number, DAS finds it. Signs are oriented so each rises with
its own number.

In [ ]:
def _dirs():
    out = {}
    # The same triples as explore_multicompare's cloud, so every cosine and probe is row-aligned
    # with the n3 versions.
    trips = MC.sample_cloud_triples(N_CLOUD, SEED + 11, LOW, HIGH)
    cloud = MC.capture_cloud(trips, T, POS2, heads=())
    yv = np.array(trips, dtype=np.int64)
    out['cloud_y'] = yv
    for k, v in cloud.items():
        out[f'cloud_{k}'] = v
    # w1, w2: rank-1 DAS on the L13 residual at n2, each fitted on the case whose number it
    # carries. Both are causal fits -- the PC1 definition of w2 was dropped because the DAS one
    # is what actually moves the answer (see explore_multicompare's u2: IIA 0.98).
    dirs = {}
    for nm, case in (('w1', 'y1'), ('w2', 'y2')):
        tbat = MC.make_batch(TRAIN_Q, CASES[case], T)
        tbat['pld_clean'], tbat['pld_corr'] = MC.pos_anchors(tbat)
        ch = S.clean_hidden_at(tbat['ids_c'], LAYER)
        Q, curve = MC.train_resid_das(LAYER, tbat, ch, POS2, 1, seed=0)
        dirs[nm] = Q[:, 0].cpu().numpy()
        out[f'{nm}_curve'] = curve
        print(f'  {nm}: L13 DAS @ n2 on {case}, loss {curve[0]:.3f} -> {curve[-1]:.3f}',
              flush=True)
        del ch, tbat
        torch.cuda.empty_cache()

    # Orient each direction so its component rises with its own number, then store.
    SRC_OF = {'w1': ('l13', 1), 'w2': ('l13', 2)}
    for nm, (ckey, j) in SRC_OF.items():
        d = dirs[nm] / np.linalg.norm(dirs[nm])
        r = np.corrcoef(cloud[ckey] @ d, np.log(yv[:, j - 1]))[0, 1]
        out[nm] = (d * np.sign(r)).astype(np.float32)
        out[f'{nm}_corr'] = np.array(abs(r))
        print(f'  {nm}: |corr| with log y{j} on {ckey} = {abs(r):.3f}', flush=True)

    # Probes: ridge log-magnitude readouts on the pre-MLP L14 cloud at n2. y3 is included as a
    # negative control -- it cannot be there, so its R2 is the floor for the other two.
    probes = MC.fit_probes(cloud['pre14'],
                           {f'p{j}': yv[:, j - 1].astype(float) for j in (1, 2, 3)})
    for nm, (w, r2) in probes.items():
        out[nm], out[f'{nm}_r2'] = w, np.array(r2)
        print(f'  probe {nm}: held-out R2 {r2:.3f}', flush=True)
    return out


DIRS = cached('dirs', _dirs)
CL = {k[6:]: DIRS[k] for k in DIRS if k.startswith('cloud_') and k != 'cloud_y'}
YV = DIRS['cloud_y']
WDIR = {1: DIRS['w1'].astype(np.float64), 2: DIRS['w2'].astype(np.float64)}
PROBE = {j: DIRS[f'p{j}'].astype(np.float64) for j in (1, 2, 3)}

# The patch / display basis: Gram-Schmidt on (w1, w2), both rank-1 DAS fits. Axis 1 IS w1.
_Wst = np.stack([WDIR[1], WDIR[2]], axis=1)
_Q, _R = np.linalg.qr(_Wst)
PB = (_Q * np.sign(np.diag(_R))).astype(np.float32)          # (d_model, 2), orthonormal
print('basis check  diag(P^T W) =', np.round(np.diag(PB.T.astype(np.float64) @ _Wst), 3))
print('cos(w1, w2) =', f'{float(WDIR[1] @ WDIR[2]):+.3f}')

## The plane

Pre-MLP L14 residual at $n_2$, projected on the orthonormalised $(\mathbf{w}_1, \mathbf{w}_2)$
frame. Gram-Schmidt gives an orthonormal basis **for the plane**; $\mathbf{w}_1$ and
$\mathbf{w}_2$ themselves are not perpendicular and both live inside it, so axis 1 is
$\mathbf{w}_1$ and axis 2 is what is left of $\mathbf{w}_2$ after removing $\mathbf{w}_1$. The
cloud's geometry inside the plane is exact; only the axis labels are loose.

In [ ]:
PQ = (CL['pre14'].astype(np.float64) - CL['pre14'].astype(np.float64).mean(0)) @ PB

fig, axes = plt.subplots(1, 3, figsize=(19, 5.6))
for ax, j in zip(axes, (1, 2, 3)):
    sc = ax.scatter(PQ[:, 0], PQ[:, 1], c=YV[:, j - 1], cmap=CMAP, s=9, alpha=0.65,
                    edgecolors='none')
    cb = fig.colorbar(sc, ax=ax, shrink=0.85, pad=0.02)
    cb.set_label(Y[j], fontsize=LABEL_FS); cb.ax.tick_params(labelsize=TICK_FS)
    cb.outline.set_visible(False)
    style_axes(ax, xlabel=W[1], ylabel=W[2] if j == 1 else None, grid=False)
    ax.grid(alpha=0.3)
    ax.set_title(f'coloured by {Y[j]}' + ('   (unreachable)' if j == 3 else ''),
                 fontsize=TITLE_FS - 2)
fig.suptitle('pre-MLP L14 residual at $n_2$ in the $(w_1, w_2)$ frame', fontsize=TITLE_FS)
plt.tight_layout(); savefig(fig, 'plane_2d'); plt.show()

for j in (1, 2, 3):
    r = [np.corrcoef(PQ[:, a], np.log(YV[:, j - 1]))[0, 1] for a in range(2)]
    print(f'log y{j} vs axes: ' + '  '.join(f'w{a + 1} {r[a]:+.3f}' for a in range(2)))

# 3. Patching the plane

Each direction alone, the two together as separate rank-1 patches, the rank-2 span at three
sites, and the full-rank ceilings. The rank-1-pair vs rank-2 comparison is the point: the span is
basis independent, the separate patches are not.

In [ ]:
def Pvec(d):
    '''A (d_model, 1) orthonormal patch basis from a single direction.'''
    return torch.tensor(np.asarray(d, dtype=np.float32), device=device).reshape(-1, 1)


def _space():
    out = {}
    Pt = torch.tensor(PB, device=device)
    P1, P2 = Pvec(DIRS['w1']), Pvec(DIRS['w2'])
    ARMS = {'w1 only':       dict(resid=[(LAYER, 'sub', P1)]),
            'w2 only':       dict(resid=[(LAYER, 'sub', P2)]),
            'w1+w2 sep':     dict(resid=[(LAYER, 'sub', P1), (LAYER, 'sub', P2)]),
            'span L13':      dict(resid=[(LAYER, 'sub', Pt)]),
            'span preMLP':   dict(premlp=Pt),
            'span postMLP':  dict(resid=[(H_LAYER, 'sub', Pt)]),
            'L13 full':      dict(resid=[(LAYER, 'full', None)]),
            'L14 full':      dict(resid=[(H_LAYER, 'full', None)]),
            'L14 full preMLP': dict(premlp=None)}
    for case in CASE_LIST:
        o = CASES[case]
        bat, src = get_bat(o), get_src(o, POS2)
        for nm, spec in ARMS.items():
            rec, iia = MC.eval_spec(spec, bat, src, POS2)
            out[f'sp_posrec_{case}_{ck(nm)}'] = rec.astype(np.float32)
            out[f'sp_iia_{case}_{ck(nm)}'] = iia.astype(np.float32)
        print(f'{case}: ' + '  '.join(f'{nm} {out[f"sp_iia_{case}_{ck(nm)}"].mean():.2f}'
                                      for nm in ARMS), flush=True)
    out['sp_arm_names'] = np.array(list(ARMS))
    out['plane_basis'] = PB
    out['plane_pq'] = PQ.astype(np.float32)
    return out


SPACE = cached('space', _space)
SP_NAMES = [str(s) for s in SPACE['sp_arm_names']]

In [ ]:
SP_JOINT = {'w1+w2 sep', 'span L13', 'span preMLP', 'span postMLP'}

fig, axes = plt.subplots(1, 2, figsize=(19, 5.8))
for ax, (key, ylab, ylim) in zip(axes, [('iia', 'IIA', (0, 1.28)),
                                        ('posrec', 'position recovery', None)]):
    xs, Wb = np.arange(len(SP_NAMES)), 0.8 / len(CASE_LIST)
    for ci, case in enumerate(CASE_LIST):
        for k, nm in enumerate(SP_NAMES):
            v = SPACE[f'sp_{key}_{case}_{ck(nm)}']
            xp = xs[k] + (ci - (len(CASE_LIST) - 1) / 2) * Wb
            ax.bar(xp, v.mean(), Wb, yerr=v.std() / np.sqrt(len(v)), capsize=2.5,
                   color=CASE_COL[case], alpha=0.92,
                   hatch=JOINT_HATCH if nm in SP_JOINT else '', edgecolor='white', linewidth=0,
                   label=f'{Y[int(case[1])]} perturbed' if k == 0 else None)
    ax.set_xticks(xs)
    ax.set_xticklabels([n.replace(' ', '\n') for n in SP_NAMES], fontsize=10)
    ax.axhline(0, color='0.6', lw=0.9)
    if ylim: ax.set_ylim(*ylim)
    style_axes(ax, ylabel=ylab, xlabel='patched component')
    ax.legend(fontsize=LEGEND_FS, loc='upper left', framealpha=0.92)
fig.suptitle('interchange at $n_2$: each direction, the rank-2 span, and the full-rank ceilings',
             fontsize=TITLE_FS)
plt.tight_layout(); savefig(fig, 'plane_patching'); plt.show()

print(f'{"arm":>20}' + ''.join(f'{c + " IIA":>12}{c + " posrec":>14}' for c in CASE_LIST))
for nm in SP_NAMES:
    print(f'{nm:>20}' + ''.join(f'{SPACE[f"sp_iia_{c}_{ck(nm)}"].mean():>12.3f}'
                                f'{SPACE[f"sp_posrec_{c}_{ck(nm)}"].mean():>14.3f}'
                                for c in CASE_LIST))

# 4. Angles between the directions, and the probes

Cosine similarity of $\mathbf{w}_1, \mathbf{w}_2$ and the ridge log-magnitude probes fitted on
the same cloud. Chance is $1/\sqrt{d_\text{model}}$.

In [ ]:
from matplotlib.patches import Rectangle

NAMES6 = [('w1', W[1]), ('w2', W[2]), ('p1', r'$\mathbf{p}_1$'), ('p2', r'$\mathbf{p}_2$')]
VECS = [DIRS[k].astype(np.float64) for k, _ in NAMES6]
LBL = [l for _, l in NAMES6]
Cm = np.array([[float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b))) for b in VECS]
               for a in VECS])
d_model = len(VECS[0])
n6 = len(NAMES6)

fig, axes = plt.subplots(1, 2, figsize=(16.5, 6.2), width_ratios=[1, 1.35])
ax = axes[0]
im = ax.imshow(np.where(np.tril(np.ones_like(Cm)) > 0, Cm, np.nan),
               cmap=matplotlib.colormaps['RdBu_r'].with_extremes(bad='white'), vmin=-1, vmax=1)
ax.set_xticks(range(n6)); ax.set_xticklabels(LBL, fontsize=15)
ax.set_yticks(range(n6)); ax.set_yticklabels(LBL, fontsize=15)
for i in range(n6):
    for j in range(i + 1):
        ax.text(j, i, f'{Cm[i, j]:+.2f}', ha='center', va='center', fontsize=12,
                color='white' if abs(Cm[i, j]) > 0.6 else 'black')
ax.add_patch(Rectangle((0 - 0.5, 1 - 0.5), 1, 1, fill=False, edgecolor='#111111', lw=2.6))
cb = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.03)
cb.set_label('cosine similarity', fontsize=LABEL_FS); cb.ax.tick_params(labelsize=TICK_FS)
ax.grid(False); ax.tick_params(length=0)
for s_ in ax.spines.values(): s_.set_visible(False)
ax.set_title('boxed: the two plane axes', fontsize=TITLE_FS - 2)

ax = axes[1]
for j in (1, 2, 3):
    comp = CL['pre14'].astype(np.float64) @ PROBE[j]
    xg, m, se = aggregate(YV[:, j - 1], comp - comp.mean())
    ax.errorbar(xg, m, yerr=se, fmt='o', ms=4.5, color=NCOL[j], linestyle='none',
                elinewidth=1.1, capsize=2, alpha=0.9,
                label=rf'$\mathbf{{p}}_{j}$ ($R^2$={float(DIRS[f"p{j}_r2"]):.2f})')
ax.axhline(0, color='0.7', lw=0.9)
style_axes(ax, xlabel='number value at that slot', ylabel='component along the probe', grid=False)
ax.grid(alpha=0.3)
ax.legend(fontsize=LEGEND_FS)
ax.set_title('pre-MLP L14 residual @ $n_2$, read by each probe', fontsize=TITLE_FS - 2)
plt.tight_layout(); savefig(fig, 'angles_probes'); plt.show()

ang = lambda i, j: np.degrees(np.arccos(np.clip(Cm[i, j], -1, 1)))
PLAIN = [k for k, _ in NAMES6]
for i in range(n6):
    for j in range(i):
        print(f'{PLAIN[i]:>6} vs {PLAIN[j]:>6}: cos {Cm[i, j]:+.3f}  ({ang(i, j):5.1f} deg)')
print(f'chance |cos| = 1/sqrt(d_model) = {1 / np.sqrt(d_model):.4f}')
print('probe held-out R2: ' + '  '.join(f'p{j} {float(DIRS[f"p{j}_r2"]):.3f}' for j in (1, 2, 3)))
print('|corr| of each direction with its own number: '
      + '  '.join(f'{k} {float(DIRS[f"{k}_corr"]):.3f}' for k in ('w1', 'w2')))

# 5. Probability contours over the plane

Both coordinates are **set** to the cloud-mean coordinate of a chosen $(y_1^*, y_2^*)$ over a
$9 \times 9$ grid at the pre-MLP L14 site, and the answered slot is recorded:

$$P_j(y_1^*, y_2^*) = \Pr[\text{the model names its own slot-}j\text{ number}]$$

The grid axes are the Gram-Schmidt axes, so $y_1^*$ and $y_2^*$ mean "the cloud-mean coordinate
on axis 1 / axis 2", not "the $\mathbf{w}_1$ / $\mathbf{w}_2$ component".

In [ ]:
SWEEP_CASE = 'y2'
SWEEP_N = 24          # prompts per grid cell


def _sweep():
    grid = MC.sweep_grid_values(LOW, HIGH)
    bat = MC._slice_batch(get_bat(CASES[SWEEP_CASE]), 0, SWEEP_N)
    coords = CL['pre14'].astype(np.float64) @ PB          # raw (uncentred) frame the patch uses
    coord_of = M2.mean_coords_k(coords, YV, grid, 2)
    ld, am = M2.sweep_plane(bat, torch.tensor(PB, device=device), coord_of, grid, POS2)
    with warnings.catch_warnings():                # the skipped diagonal is an all-nan slice
        warnings.simplefilter('ignore', RuntimeWarning)
        ld_mean = np.nanmean(ld, axis=-1)
    tsl = bat['tok_slot'].cpu().numpy()
    P = np.stack([(am == tsl[:, j][None, None, :]).mean(-1) for j in range(3)])   # (3, G, G)
    named = np.stack([(am == tsl[:, j][None, None, :]) for j in range(3)]).any(0).mean(-1)
    return dict(grid=np.array(grid), P=P.astype(np.float32), named=named.astype(np.float32),
                ld=ld_mean.astype(np.float32),
                coord_of=np.stack([coord_of[y] for y in grid]).astype(np.float32),
                case=np.array(SWEEP_CASE), n_prompts=np.array(SWEEP_N))


SWEEP = cached('sweep', _sweep)
GRID = SWEEP['grid']
NAMED = SWEEP['named'].astype(np.float64)
VALID = NAMED > 0                                         # False on the skipped diagonal
PFIELD = np.where(VALID[None], SWEEP['P'].astype(np.float64), np.nan)
PCOND = PFIELD / np.where(VALID, NAMED, np.nan)[None]
print(f"grid {GRID.tolist()}   case {SWEEP['case']}   {int(SWEEP['n_prompts'])} prompts/cell")
print(f'the model named one of its own three numbers in {NAMED[VALID].mean():.3f} of '
      f'scored prompts ({VALID.sum()} of {NAMED.size} cells scored)')

In [ ]:
SLOT_COL3 = [NCOL[1], NCOL[2], NCOL[3]]
cmap3 = matplotlib.colors.ListedColormap(SLOT_COL3)
_h = 0.5 * (GRID[1] - GRID[0])
ext = [GRID[0] - _h, GRID[-1] + _h, GRID[0] - _h, GRID[-1] + _h]

fig, axes = plt.subplots(1, 3, figsize=(19, 5.4))

ax = axes[0]
ok = np.isfinite(PCOND[0])
win = np.where(ok, np.nanargmax(np.where(ok[None], PCOND, -1), axis=0).astype(float), np.nan)
ax.imshow(np.ma.masked_invalid(win).T, origin='lower', extent=ext, cmap=cmap3, vmin=-0.5,
          vmax=2.5, alpha=0.32, aspect='auto', interpolation='nearest')
for j in range(3):
    f = np.where(ok, PCOND[j], np.nan)
    if np.nanmax(f) > 0.5 > np.nanmin(f):
        ax.contour(GRID, GRID, f.T, levels=[0.5], colors=[SLOT_COL3[j]], linewidths=2.2)
ax.plot(GRID, GRID, ls=':', color='0.35', lw=1.2)
style_axes(ax, xlabel='$y_1^*$', ylabel='$y_2^*$', grid=False)
ax.set_title('argmax slot (fill), $P = 0.5$ (lines)', fontsize=TITLE_FS - 2)

for ax, (j, ttl) in zip(axes[1:], [(0, 'P(names slot 1)'), (1, 'P(names slot 2)')]):
    im = ax.imshow(np.ma.masked_invalid(PCOND[j]).T, origin='lower', extent=ext, cmap='viridis',
                   vmin=0, vmax=1, aspect='auto', interpolation='nearest')
    ax.plot(GRID, GRID, ls=':', color='w', lw=1.2)
    cb = fig.colorbar(im, ax=ax, shrink=0.85, pad=0.02); cb.ax.tick_params(labelsize=TICK_FS)
    style_axes(ax, xlabel='$y_1^*$', ylabel='$y_2^*$', grid=False)
    ax.set_title(ttl, fontsize=TITLE_FS - 2)

fig.suptitle(f'implanted $(w_1, w_2)$ coordinates at $n_2$  (case {str(SWEEP["case"])})',
             fontsize=TITLE_FS)
plt.tight_layout(); savefig(fig, 'plane_contours'); plt.show()

hit = n = 0
for i, y1 in enumerate(GRID):
    for j, y2 in enumerate(GRID):
        if not VALID[i, j]:
            continue
        hit += int(np.argmax(PCOND[:2, i, j]) == int(np.argmax([y1, y2])))
        n += 1
print(f'the named slot matches the larger implanted number in {hit}/{n} ({hit / n:.3f}) '
      f'of scored cells  (chance 0.5, restricted to slots 1 and 2)')
print('mean PLD over the plane: '
      f'{np.nanmin(SWEEP["ld"]):+.2f} to {np.nanmax(SWEEP["ld"]):+.2f}')

# 6. Freezing MLP and attention at layers 14, 15, 16

Base patch is the rank-2 span at the pre-MLP L14 residual. A freeze pins a sub-block's output at
$n_2$ to its unpatched corrupted value, so the patch cannot reach the rest of the network through
it. This says which sub-block the plane acts through, before any neuron is looked at.

In [ ]:
SWEEP_LAYERS = [14, 15, 16]
FREEZE_SETS = {'none': ()}
FREEZE_SETS.update({f'MLP{l}': (('mlp', l),) for l in SWEEP_LAYERS})
FREEZE_SETS.update({f'attn{l}': (('attn', l),) for l in SWEEP_LAYERS})
FREEZE_SETS['MLP14+15'] = (('mlp', 14), ('mlp', 15))
FREEZE_SETS['MLP14+15+16'] = (('mlp', 14), ('mlp', 15), ('mlp', 16))
SITES = [(k, l) for l in SWEEP_LAYERS for k in ('mlp', 'attn')]

FRZ = {}


def get_frz(o):
    if o not in FRZ:
        FRZ[o] = C.capture_sites(SITES, get_bat(o)['ids_r'], POS2)
    return FRZ[o]


def freeze_extra(o, fz):
    src = get_frz(o)
    return lambda lo, hi: [C.block_of(s).register_forward_hook(
        C.make_out_patch_hook(src[s][lo:hi], POS2)) for s in fz]


def _freeze():
    out = {}
    Pt = torch.tensor(PB, device=device)
    for case in CASE_LIST:
        o = CASES[case]
        bat, src = get_bat(o), get_src(o, POS2)
        for nm, fz in FREEZE_SETS.items():
            rec, iia = MC.eval_spec(dict(premlp=Pt), bat, src, POS2, extra=freeze_extra(o, fz))
            out[f'posrec_{case}_{nm}'] = rec.astype(np.float32)
            out[f'iia_{case}_{nm}'] = iia.astype(np.float32)
        print(f'{case}: ' + '  '.join(f'{nm} {out[f"iia_{case}_{nm}"].mean():.2f}'
                                      for nm in FREEZE_SETS), flush=True)
    out['freeze_names'] = np.array(list(FREEZE_SETS))
    return out


FREEZE = cached('freeze', _freeze)

In [ ]:
fz_names = [str(s) for s in FREEZE['freeze_names']]
fig, axes = plt.subplots(1, 2, figsize=(17, 5.6))
for ax, (key, ylab, ylim) in zip(axes, [('iia', 'IIA', (0, 1.15)),
                                        ('posrec', 'position recovery', None)]):
    xs, Wb = np.arange(len(fz_names)), 0.8 / len(CASE_LIST)
    for ci, case in enumerate(CASE_LIST):
        v = np.array([FREEZE[f'{key}_{case}_{nm}'].mean() for nm in fz_names])
        e = np.array([FREEZE[f'{key}_{case}_{nm}'].std()
                      / np.sqrt(len(FREEZE[f'{key}_{case}_{nm}'])) for nm in fz_names])
        ax.bar(xs + (ci - (len(CASE_LIST) - 1) / 2) * Wb, v, Wb, yerr=e, capsize=3,
               color=CASE_COL[case], alpha=0.92, label=f'{Y[int(case[1])]} perturbed')
        ax.axhline(FREEZE[f'{key}_{case}_none'].mean(), ls='--', lw=1.1,
                   color=CASE_COL[case], alpha=0.7)
    ax.set_xticks(xs); ax.set_xticklabels(fz_names, rotation=30, ha='right', fontsize=TICK_FS)
    ax.axhline(0, color='0.6', lw=0.9)
    if ylim: ax.set_ylim(*ylim)
    style_axes(ax, ylabel=ylab, xlabel='frozen sub-block')
    ax.legend(fontsize=LEGEND_FS)
fig.suptitle('freezing under the rank-2 span patch at $n_2$ (dashed: that case unfrozen)',
             fontsize=TITLE_FS)
plt.tight_layout(); savefig(fig, 'module_freezes'); plt.show()

In [ ]:
# The neuron set is taken from the TWO-NUMBER comparator, not re-derived here: the 12 neurons
# `makefig_comparator` found shared between its two cases (`overlap_k20`). Attribution patching
# at n2 is skipped -- these are the neurons of interest, and the question is how they behave here.
N_POOLED = 2 * D_FF
SEL = np.asarray(TWO['overlap_k20']) if TWO is not None else np.array([], dtype=np.int64)
RANKS = None                      # no attribution ranking to annotate the fields with

NEUR = {}


def get_neur(o):
    if o not in NEUR:
        NEUR[o] = C.capture_neurons(get_bat(o)['ids_r'], NEURON_LAYERS, POS2)
    return NEUR[o]


def neuron_sel(pooled):
    pooled = np.asarray(pooled)
    return {l: torch.as_tensor(POOL_WITHIN[pooled[POOL_LAYER[pooled] == l]],
                               device=device, dtype=torch.long) for l in NEURON_LAYERS}


def freeze_neurons(case, pooled, base=None):
    '''IIA / position recovery with `pooled` neurons pinned to their corrupted values.'''
    o = CASES[case]
    sel, corr = neuron_sel(pooled), get_neur(o)
    Pt = torch.tensor(PB, device=device)

    def extra(lo, hi):
        return [C.mlp_of(l).down_proj.register_forward_pre_hook(
            C.make_neuron_freeze_hook(ii, corr[l][lo:hi][:, ii], POS2))
            for l, ii in sel.items() if len(ii)]
    return MC.eval_spec(base if base is not None else dict(premlp=Pt),
                        get_bat(o), get_src(o, POS2), POS2, extra=extra)


print(f'{len(SEL)} two-number neurons carried over  '
      f'(L14 {(POOL_LAYER[SEL] == 14).sum()}, L15 {(POOL_LAYER[SEL] == 15).sum()})')
print('  ' + ', '.join(NAME(p) for p in SEL))


# 7. Receptive fields at $n_2$

Post-SwiGLU activation over a dense $(y_1, y_2)$ grid, $y_3$ pinned. One surface per neuron —
at $n_2$ there is no third axis to average out.

In [ ]:
RF_STEP = 4
RF_GRID = np.arange(LOW + 1, HIGH, RF_STEP)
RF_Y3 = LOW + (HIGH - LOW) // 2                       # pinned; unreachable from n2 anyway

TRANSFER = SEL
RF_POOLED = np.array(sorted(set(np.asarray(SEL).tolist())), dtype=np.int64)


def _rf():
    idx = {l: torch.as_tensor(POOL_WITHIN[RF_POOLED[POOL_LAYER[RF_POOLED] == l]],
                              device=device, dtype=torch.long) for l in NEURON_LAYERS}
    pairs, acts = M2.receptive_fields_2d(RF_GRID, RF_Y3, T, POS2, NEURON_LAYERS, idx)
    out = {'rf_grid': RF_GRID.astype(np.int64), 'rf_pooled': RF_POOLED, 'rf_pairs': pairs,
           'rf_y3': np.array(RF_Y3)}
    for l in NEURON_LAYERS:
        out[f'rf_acts_L{l}'] = acts[l]
        out[f'rf_idx_L{l}'] = idx[l].cpu().numpy()
    return out


RF = cached('rf', _rf)
G = len(RF['rf_grid'])
RF_COL = {p: (int(POOL_LAYER[p]), int(np.where(RF[f'rf_idx_L{POOL_LAYER[p]}']
                                               == POOL_WITHIN[p])[0][0])) for p in RF['rf_pooled']}


def rf_field(pooled):
    '''(G, G) activation surface for one pooled neuron, axes (y1, y2).'''
    l, c = RF_COL[pooled]
    return RF[f'rf_acts_L{l}'][:, c].reshape(G, G)


def draw_fields(pooled_list, fname, title, ranks=None, ncol=4):
    '''One (y1, y2) heatmap per neuron, diverging scale, one scale per neuron.'''
    n = len(pooled_list)
    nc = min(ncol, n)
    nr = int(np.ceil(n / nc))
    g = RF['rf_grid']
    h = 0.5 * (g[1] - g[0])
    ext = [g[0] - h, g[-1] + h, g[0] - h, g[-1] + h]
    fig, axes = plt.subplots(nr, nc, figsize=(3.6 * nc, 3.5 * nr), squeeze=False)
    for slot in range(nr * nc):
        ax = axes[slot // nc, slot % nc]
        if slot >= n:
            ax.axis('off'); continue
        p = int(pooled_list[slot])
        f = rf_field(p)
        vmax = np.abs(f).max() + 1e-9
        im = ax.imshow(f.T, origin='lower', extent=ext, cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                       aspect='auto', interpolation='nearest')
        ax.plot(g, g, ls=':', color='0.3', lw=0.9)
        ax.set_xlabel(Y[1], fontsize=12, labelpad=1)
        ax.set_ylabel(Y[2], fontsize=12, labelpad=0)
        ax.tick_params(labelsize=8)
        rk = '' if ranks is None else f" (rank {ranks.get(p, '-')})"
        ax.set_title(f'{NAME(p)}{rk}', fontsize=13, fontweight='bold')
        fig.colorbar(im, ax=ax, shrink=0.85, pad=0.02).ax.tick_params(labelsize=8)
    fig.suptitle(title + f'   ($y_3$ pinned at {int(RF["rf_y3"])}, one scale per neuron)',
                 fontsize=TITLE_FS)
    plt.tight_layout(); savefig(fig, fname); plt.show()

In [ ]:
draw_fields(list(SEL), 'rf_two_number_neurons',
            'the two-number comparator neurons, read at $n_2$')


# 8. Freezing the two-number neurons at $n_2$

The same 12 neurons, pinned to their corrupted values under the rank-2 span patch, against random
sets of the same size.

In [ ]:
def _transfer():
    out = {}
    if not len(TRANSFER):
        return {'none': np.array(0)}
    rng = np.random.default_rng(7)
    sets = {'two_number': np.asarray(TRANSFER),
            'rand0': rng.choice(N_POOLED, len(TRANSFER), replace=False),
            'rand1': rng.choice(N_POOLED, len(TRANSFER), replace=False)}
    for case in CASE_LIST:
        for nm, pooled in sets.items():
            rec, iia = freeze_neurons(case, pooled)
            out[f'posrec_{case}_{nm}'] = rec.astype(np.float32)
            out[f'iia_{case}_{nm}'] = iia.astype(np.float32)
        print(f'{case}: two-number set {out[f"iia_{case}_two_number"].mean():.3f}   '
              f'random {out[f"iia_{case}_rand0"].mean():.3f}   '
              f'unfrozen {FREEZE[f"iia_{case}_none"].mean():.3f}', flush=True)
    out['pooled'] = np.asarray(TRANSFER)
    return out


TRANS = cached('transfer', _transfer)

In [ ]:
if len(TRANSFER):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
    for ax, (key, ylab, ylim) in zip(axes, [('iia', 'IIA', (0, 1.12)),
                                            ('posrec', 'position recovery', None)]):
        xs, Wb = np.arange(len(CASE_LIST)), 0.26
        bars = [('none', 'no freeze', COL['ceil']),
                ('two_number', f'the {len(TRANSFER)} two-number neurons', COL['full']),
                ('rand0', f'{len(TRANSFER)} random neurons', COL['pc1'])]
        for bi, (nm, lab, col) in enumerate(bars):
            vals = [(FREEZE[f'{key}_{c}_none'] if nm == 'none' else TRANS[f'{key}_{c}_{nm}'])
                    for c in CASE_LIST]
            ax.bar(xs + (bi - 1) * Wb, [v.mean() for v in vals], Wb,
                   yerr=[v.std() / np.sqrt(len(v)) for v in vals], capsize=4, color=col,
                   alpha=0.92, label=lab)
        ax.set_xticks(xs)
        ax.set_xticklabels([f'{Y[int(c[1])]} perturbed' for c in CASE_LIST], fontsize=LABEL_FS)
        ax.axhline(0, color='0.6', lw=0.9)
        if ylim: ax.set_ylim(*ylim)
        style_axes(ax, ylabel=ylab)
        ax.legend(fontsize=LEGEND_FS)
    fig.suptitle('freezing the two-number neurons under the $n_2$ span patch', fontsize=TITLE_FS)
    plt.tight_layout(); savefig(fig, 'transfer_freeze'); plt.show()

# 9. L15 at $n_2$ — the partial comparison

Rank-1 DAS on the residual leaving L15 at $n_2$, and the PCA of the same cloud, every projection
coloured by **which of $y_1, y_2$ is larger** — the only comparison available at this position.

In [ ]:
DAS15_LAYER = 15
DAS15_RANKS = [1]              # add 2 for the 2-D DAS scatter below; costs ~13 min more
DAS15_CASES = CASE_LIST
DAS15_SEED = 0


def src_with_L15(o):
    '''Section 4's sources plus the L15 residual; `get_src` caches per layer, so L13 and L14 are
    not re-captured.'''
    return get_src(o, POS2, layers=(LAYER, H_LAYER, DAS15_LAYER))


def _das15():
    out = {}
    trips = MC.sample_cloud_triples(N_CLOUD, SEED + 11, LOW, HIGH)
    assert np.array_equal(np.array(trips, dtype=np.int64), YV), \
        'the L15 cloud must be the same triples as the section-3 cloud'
    c15 = MC.capture_resid_cloud(trips, T, [(DAS15_LAYER, POS2)])[(DAS15_LAYER, POS2)]
    out['cloud15'] = c15

    _, pca15 = MC.pc1_of(c15)
    out['pca15_coords'] = pca15.transform(c15).astype(np.float32)
    out['pca15_components'] = pca15.components_.astype(np.float32)
    out['pca15_evr'] = pca15.explained_variance_ratio_.astype(np.float32)
    print(f'  L15 PCA EVR {np.round(pca15.explained_variance_ratio_, 3).tolist()}', flush=True)

    bases = {}
    for case in DAS15_CASES:
        tbat = MC.make_batch(TRAIN_Q, CASES[case], T)
        tbat['pld_clean'], tbat['pld_corr'] = MC.pos_anchors(tbat)
        ch = S.clean_hidden_at(tbat['ids_c'], DAS15_LAYER)
        for k in DAS15_RANKS:
            Q, curve = MC.train_resid_das(DAS15_LAYER, tbat, ch, POS2, k, seed=DAS15_SEED)
            bases[(case, k)] = Q
            out[f'basis_{case}_k{k}'] = Q.cpu().numpy().astype(np.float32)
            out[f'curve_{case}_k{k}'] = curve
            out[f'proj_{case}_k{k}'] = (c15 @ Q.cpu().numpy()).astype(np.float32)
            print(f'  L15 DAS {case} k={k}: loss {curve[0]:.3f} -> {curve[-1]:.3f}', flush=True)
            for tc in CASE_LIST:
                rec, iia = MC.eval_spec(dict(resid=[(DAS15_LAYER, 'sub', Q)]),
                                        get_bat(CASES[tc]), src_with_L15(CASES[tc]), POS2)
                out[f'posrec_{tc}_from_{case}_k{k}'] = rec.astype(np.float32)
                out[f'iia_{tc}_from_{case}_k{k}'] = iia.astype(np.float32)
        del ch, tbat
        torch.cuda.empty_cache()

    for tc in CASE_LIST:                       # full-rank ceiling at the same site
        rec, iia = MC.eval_spec(dict(resid=[(DAS15_LAYER, 'full', None)]),
                                get_bat(CASES[tc]), src_with_L15(CASES[tc]), POS2)
        out[f'posrec_{tc}_full'] = rec.astype(np.float32)
        out[f'iia_{tc}_full'] = iia.astype(np.float32)

    out.update(C.das15_write_alignment(bases, layer=DAS15_LAYER))
    return out


DAS15 = cached('das15', _das15)
BIGGER = (YV[:, 1] > YV[:, 0]).astype(int)      # 0 = y1 larger, 1 = y2 larger
print('cloud counts (y1 larger, y2 larger):', np.bincount(BIGGER, minlength=2).tolist())

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

DAS15_SHOW = 'y2'                 # which fitted subspace to draw
BIGC = [NCOL[1], NCOL[2]]
BIG_LAB = {0: '$y_1 > y_2$', 1: '$y_2 > y_1$'}

p1 = DAS15[f'proj_{DAS15_SHOW}_k1'][:, 0]
HAS_K2 = 2 in DAS15_RANKS
pc, evr = DAS15['pca15_coords'], DAS15['pca15_evr']

npan = 4 if HAS_K2 else 3
fig, axes = plt.subplots(1, npan, figsize=(5.5 * npan, 5.2))
axes = list(axes)

ax = axes.pop(0)
bins = np.linspace(p1.min(), p1.max(), 46)
for s in (0, 1):
    ax.hist(p1[BIGGER == s], bins=bins, color=BIGC[s], alpha=0.55, label=BIG_LAB[s])
style_axes(ax, xlabel='rank-1 DAS component', ylabel='prompts', grid=False)
ax.grid(alpha=0.3); ax.legend(fontsize=LEGEND_FS)
ax.set_title(f'L15 DAS $k=1$ (fit {DAS15_SHOW})', fontsize=TITLE_FS - 3)

if HAS_K2:
    p2 = DAS15[f'proj_{DAS15_SHOW}_k2']
    ax = axes.pop(0)
    for s in (0, 1):
        m = BIGGER == s
        ax.scatter(p2[m, 0], p2[m, 1], s=9, color=BIGC[s], alpha=0.6, edgecolors='none',
                   label=BIG_LAB[s])
    style_axes(ax, xlabel='DAS 1', ylabel='DAS 2', grid=False)
    ax.grid(alpha=0.3); ax.legend(fontsize=LEGEND_FS, markerscale=2)
    ax.set_title(f'L15 DAS $k=2$ (fit {DAS15_SHOW})', fontsize=TITLE_FS - 3)

ax = axes.pop(0)
for s in (0, 1):
    m = BIGGER == s
    ax.scatter(pc[m, 0], pc[m, 1], s=9, color=BIGC[s], alpha=0.6, edgecolors='none')
style_axes(ax, xlabel=f'PC1 ({100 * evr[0]:.0f}%)', ylabel=f'PC2 ({100 * evr[1]:.0f}%)',
           grid=False)
ax.grid(alpha=0.3)
ax.set_title('L15 PCA', fontsize=TITLE_FS - 3)

ax = axes.pop(0)
sc = ax.scatter(p1, np.log(YV[:, 1]) - np.log(YV[:, 0]), s=9, c=YV[:, 1], cmap=CMAP, alpha=0.7,
                edgecolors='none')
ax.axhline(0, color='0.5', lw=1.0)
cb = fig.colorbar(sc, ax=ax, shrink=0.85, pad=0.02); cb.set_label(Y[2], fontsize=LABEL_FS)
cb.ax.tick_params(labelsize=TICK_FS); cb.outline.set_visible(False)
style_axes(ax, xlabel='rank-1 DAS component', ylabel=r'$\log y_2 - \log y_1$', grid=False)
ax.grid(alpha=0.3)
ax.set_title('the DAS component vs the contrast', fontsize=TITLE_FS - 3)

fig.suptitle('the L15 residual at $n_2$, coloured by which of $y_1, y_2$ is larger',
             fontsize=TITLE_FS)
plt.tight_layout(); savefig(fig, 'das15_spaces'); plt.show()

READOUTS = [('DAS k=1', p1.reshape(-1, 1))]
if HAS_K2:
    READOUTS.append(('DAS k=2', p2))
READOUTS += [('PCA top 2', pc[:, :2]), ('PCA top 3', pc[:, :3])]
for nm, X in READOUTS:
    Xtr, Xte, ytr, yte = train_test_split(X, BIGGER, test_size=0.25, random_state=SEED,
                                          stratify=BIGGER)
    acc = LogisticRegression(max_iter=2000).fit(Xtr, ytr).score(Xte, yte)
    print(f'"y2 larger", held-out accuracy from {nm:>10}: {acc:.3f}   (chance 0.500)')
r_ = np.corrcoef(p1, np.log(YV[:, 1]) - np.log(YV[:, 0]))[0, 1]
print(f'corr(rank-1 DAS component, log y2 - log y1) = {r_:+.3f}')

In [ ]:
fig, axes = plt.subplots(1, len(DAS15_RANKS) + 1,
                         figsize=(5.6 * (len(DAS15_RANKS) + 1), 5.0),
                         width_ratios=[1] * len(DAS15_RANKS) + [1.25])
for ax, k in zip(axes, DAS15_RANKS):
    Mx = np.array([[DAS15[f'iia_{tc}_from_{fc}_k{k}'].mean() for tc in CASE_LIST]
                   for fc in DAS15_CASES])
    im = ax.imshow(Mx, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(len(CASE_LIST)))
    ax.set_xticklabels([f'{Y[int(c[1])]} pert.' for c in CASE_LIST], fontsize=TICK_FS)
    ax.set_yticks(range(len(DAS15_CASES)))
    ax.set_yticklabels([f'fit on {c}' for c in DAS15_CASES], fontsize=TICK_FS)
    for i in range(Mx.shape[0]):
        for j in range(Mx.shape[1]):
            ax.text(j, i, f'{Mx[i, j]:.2f}', ha='center', va='center', fontsize=13,
                    color='white' if Mx[i, j] > 0.55 else 'black')
    ax.set_title(f'L15 DAS rank {k} — IIA', fontsize=TITLE_FS - 2)
    ax.grid(False); ax.tick_params(length=0)
    for s_ in ax.spines.values():
        s_.set_visible(False)

ax = axes[-1]
xs, Wb = np.arange(len(CASE_LIST)), 0.8 / (len(DAS15_RANKS) + 2)
series = [(f'k={k} (own case)', lambda c, _k=k: DAS15[f'iia_{c}_from_{c}_k{_k}'],
           [COL['h14'], COL['pc1'], COL['das']][i % 3]) for i, k in enumerate(DAS15_RANKS)]
series.append(('L15 full', lambda c: DAS15[f'iia_{c}_full'], COL['ceil']))
series.append(('L13 span @ $n_2$', lambda c: SPACE[f'sp_iia_{c}_{ck("span preMLP")}'],
               COL['full']))
for bi, (lab, get, col) in enumerate(series):
    vals = [get(c) for c in CASE_LIST]
    ax.bar(xs + (bi - (len(series) - 1) / 2) * Wb, [v.mean() for v in vals], Wb,
           yerr=[v.std() / np.sqrt(len(v)) for v in vals], capsize=4, color=col, alpha=0.92,
           label=lab)
ax.set_xticks(xs)
ax.set_xticklabels([f'{Y[int(c[1])]} perturbed' for c in CASE_LIST], fontsize=LABEL_FS - 2)
ax.set_ylim(0, 1.15)
style_axes(ax, ylabel='IIA')
ax.legend(fontsize=LEGEND_FS - 1)
ax.set_title('rank vs the ceilings at $n_2$', fontsize=TITLE_FS - 2)
plt.tight_layout(); savefig(fig, 'das15_iia'); plt.show()

for k in DAS15_RANKS:
    for fc in DAS15_CASES:
        print(f'rank {k}, fitted on {fc}: ' + '  '.join(
            f'{tc} {DAS15[f"iia_{tc}_from_{fc}_k{k}"].mean():.3f}' for tc in CASE_LIST))

### Which MLP15 neurons write along the DAS space

$\rho_j = \lVert Q^\top w_j\rVert / \lVert w_j\rVert$, $w_j$ the neuron's `down_proj` column.
The null is every MLP15 neuron; the carried-over ones are marked in it.

In [ ]:
AL_CASE, AL_K = 'y2', 1           # which fitted subspace the alignment is measured against
N_ALIGN_SHOW = 4

frac = DAS15[f'dasfrac_L{DAS15_LAYER}_{ck(AL_CASE)}_k{AL_K}'].astype(np.float64)
prof15 = SEL[POOL_LAYER[SEL] == DAS15_LAYER]
w15 = POOL_WITHIN[prof15]
pf = frac[w15]
q95, q99 = np.quantile(frac, [0.95, 0.99])
order15 = np.argsort(-pf)
ALIGNED = prof15[order15[:N_ALIGN_SHOW]]

fig, ax = plt.subplots(figsize=(10.5, 5.2))
ax.hist(frac, bins=140, color='0.72')
ax.axvline(q95, ls='--', color='0.35', lw=1.3, label='95th pct of all MLP15')
ax.axvline(q99, ls=':', color='0.35', lw=1.3, label='99th pct')
ax.set_yscale('log')
tx = ax.get_xaxis_transform()
for r, p in enumerate(prof15[order15]):
    v = frac[POOL_WITHIN[p]]
    col = VCOL[1] if r < N_ALIGN_SHOW else COL['pc1']
    ax.axvline(v, color=col, lw=2.0 if r < N_ALIGN_SHOW else 1.0,
               alpha=0.95 if r < N_ALIGN_SHOW else 0.5)
    if r < N_ALIGN_SHOW:
        ax.text(v, 0.96 - 0.075 * r, f' {NAME(p)}', color=col, fontsize=11,
                fontweight='bold', va='top', transform=tx)
style_axes(ax, xlabel=r'$\|Q^\top w_j\| / \|w_j\|$', ylabel='MLP15 neurons', grid=False)
ax.grid(alpha=0.3)
ax.legend(fontsize=LEGEND_FS)
ax.set_title(f'write alignment with the rank-{AL_K} L15 DAS space at $n_2$ (fit {AL_CASE})',
             fontsize=TITLE_FS - 1)
plt.tight_layout(); savefig(fig, 'das15_alignment'); plt.show()

print(f'null over all {len(frac)} MLP15 neurons: median {np.median(frac):.4f}   '
      f'95th {q95:.4f}   99th {q99:.4f}')
print(f'{"neuron":>12}{"rho":>9}{"pct":>8}')
for p in prof15[order15]:
    v = frac[POOL_WITHIN[p]]
    print(f'{NAME(p):>12}{v:>9.4f}{100 * (frac < v).mean():>7.1f}%')

In [ ]:
if len(ALIGNED):
    draw_fields([int(p) for p in ALIGNED], 'rf_das15_aligned',
                f'MLP15 neurons most aligned with the rank-{AL_K} L15 DAS space at $n_2$')

# 10. Cosines across positions

$\mathbf{u}_1, \mathbf{u}_2$ (L13 at $n_1$ / $n_2$) and $\mathbf{v}_1, \mathbf{v}_2, \mathbf{v}_3$
(the $n_3$ frame) are read from `explore_multicompare`'s cache. $\mathbf{w}_2$ and $\mathbf{u}_2$
are the same fit, so their cosine is the consistency check.

In [ ]:
DSET = [(r'$\mathbf{u}_1$', 'L13 @ $n_1$', N3['ucos'], 'u1'),
        (r'$\mathbf{u}_2$', 'L13 @ $n_2$', N3['ucos'], 'u2'),
        (r'$\mathbf{w}_1$', 'L13 DAS @ $n_2$ (y1)', DIRS, 'w1'),
        (r'$\mathbf{w}_2$', 'L13 DAS @ $n_2$ (y2)', DIRS, 'w2'),
        (r'$\mathbf{v}_1$', 'H14 out @ $n_3$', N3['dirs'], 'v1'),
        (r'$\mathbf{v}_2$', 'H18 out @ $n_3$', N3['dirs'], 'v2'),
        (r'$\mathbf{v}_3$', 'L13 @ $n_3$', N3['dirs'], 'v3')]
DSET = [d for d in DSET if d[2] is not None]
VEC = [np.asarray(src[key], dtype=np.float64) for _, _, src, key in DSET]
PLAINN = [key for _, _, _, key in DSET]
nD = len(DSET)
Cd = np.array([[float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b))) for b in VEC]
               for a in VEC])

fig, axes = plt.subplots(1, 2, figsize=(19, 7.0), width_ratios=[1.3, 1])

ax = axes[0]
im = ax.imshow(np.where(np.tril(np.ones_like(Cd)) > 0, Cd, np.nan),
               cmap=matplotlib.colormaps['RdBu_r'].with_extremes(bad='white'), vmin=-1, vmax=1)
ax.set_xticks(range(nD)); ax.set_xticklabels([d[0] for d in DSET], fontsize=16)
ax.set_yticks(range(nD))
ax.set_yticklabels([f'{d[0]}  {d[1]}' for d in DSET], fontsize=12)
for i in range(nD):
    for j in range(i + 1):
        ax.text(j, i, f'{Cd[i, j]:+.2f}', ha='center', va='center', fontsize=11,
                color='white' if abs(Cd[i, j]) > 0.6 else 'black')
# Boxed: the same variable read at two places -- u2 vs its refit here, and y1 at n1 / n2 / n3.
for a_, b_ in [('u2', 'w2'), ('u1', 'w1'), ('u1', 'v1'), ('w1', 'v1')]:
    if a_ in PLAINN and b_ in PLAINN:
        i, j = PLAINN.index(a_), PLAINN.index(b_)
        ax.add_patch(Rectangle((min(i, j) - 0.5, max(i, j) - 0.5), 1, 1, fill=False,
                               edgecolor='#111111', lw=2.4, zorder=6))
cb = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.03)
cb.set_label('cosine similarity', fontsize=LABEL_FS); cb.ax.tick_params(labelsize=TICK_FS)
ax.grid(False); ax.tick_params(length=0)
for s_ in ax.spines.values(): s_.set_visible(False)
ax.set_title('boxed: the same number read at two positions', fontsize=TITLE_FS - 1)

ax = axes[1]
bars = [(W[1] + ' @ $n_2$', lambda c: SPACE[f'sp_iia_{c}_{ck("w1 only")}'], VCOL[1]),
        (W[2] + ' @ $n_2$', lambda c: SPACE[f'sp_iia_{c}_{ck("w2 only")}'], VCOL[2]),
        ('rank-2 span @ $n_2$', lambda c: SPACE[f'sp_iia_{c}_{ck("span preMLP")}'], COL['full'])]
if N3['ucos'] is not None:
    bars.append((r'$\mathbf{u}_j$ @ its own $n_j$',
                 lambda c: N3['ucos'][f'iia_{c}_u{c[1]}'], COL['ceil']))
xs, Wb = np.arange(len(CASE_LIST)), 0.8 / len(bars)
for bi, (lab, get, col) in enumerate(bars):
    vals = [np.asarray(get(c)) for c in CASE_LIST]
    ax.bar(xs + (bi - (len(bars) - 1) / 2) * Wb, [v.mean() for v in vals], Wb,
           yerr=[v.std() / np.sqrt(len(v)) for v in vals], capsize=4, color=col, alpha=0.92,
           label=lab)
ax.set_xticks(xs)
ax.set_xticklabels([f'{Y[int(c[1])]} perturbed' for c in CASE_LIST], fontsize=LABEL_FS - 1)
ax.set_ylim(0, 1.15)
style_axes(ax, ylabel='IIA')
ax.legend(fontsize=LEGEND_FS - 1)
ax.set_title('what is causal at $n_2$, against each number\'s own position', fontsize=TITLE_FS - 1)
plt.tight_layout(); savefig(fig, 'position_cosines'); plt.show()

d_m = len(VEC[0])
for i in range(nD):
    for j in range(i):
        print(f'{PLAINN[i]:>6} vs {PLAINN[j]:>6}: cos {Cd[i, j]:+.3f}  '
              f'({np.degrees(np.arccos(np.clip(Cd[i, j], -1, 1))):5.1f} deg)')
print(f'chance |cos| = 1/sqrt(d_model) = {1 / np.sqrt(d_m):.4f}')

In [ ]:
# --- how w1 and w2 (both rank-1 DAS on L13 @ n2) read the numbers -------------------------
L13 = CL['l13'].astype(np.float64)
PROJ = {1: L13 @ WDIR[1], 2: L13 @ WDIR[2]}

fig, axes = plt.subplots(1, 3, figsize=(19, 5.2))

# each axis against all three numbers; only its own should slope, y3 must be flat
for ax, k in zip(axes[:2], (1, 2)):
    comp = PROJ[k] - PROJ[k].mean()
    for j in (1, 2, 3):
        xg, m, se = aggregate(YV[:, j - 1], comp)
        ax.errorbar(xg, m, yerr=se, fmt='o' if j == k else 'x', ms=4.5, color=NCOL[j],
                    linestyle='none', elinewidth=1.1, capsize=2,
                    alpha=0.9 if j == k else 0.4,
                    label=f'vs {Y[j]}' + (' (own)' if j == k else ''))
    ax.axhline(0, color='0.7', lw=0.9)
    style_axes(ax, xlabel='number value at that slot',
               ylabel='component (centred)' if k == 1 else None, grid=False)
    ax.grid(alpha=0.3); ax.legend(fontsize=LEGEND_FS - 1)
    ax.set_title(f'{W[k]}  (|corr| with log y{k} = {float(DIRS[f"w{k}_corr"]):.3f})',
                 fontsize=TITLE_FS - 2)

# the plane itself, as the two raw components
ax = axes[2]
sc = ax.scatter(PROJ[1], PROJ[2], s=9, c=YV[:, 1], cmap=CMAP, alpha=0.7, edgecolors='none')
cb = fig.colorbar(sc, ax=ax, shrink=0.85, pad=0.02)
cb.set_label(Y[2], fontsize=LABEL_FS); cb.ax.tick_params(labelsize=TICK_FS)
cb.outline.set_visible(False)
style_axes(ax, xlabel=f'{W[1]} component', ylabel=f'{W[2]} component', grid=False)
ax.grid(alpha=0.3)
ax.set_title(f'cos(w1, w2) = {float(WDIR[1] @ WDIR[2]):+.3f}', fontsize=TITLE_FS - 2)

fig.suptitle('the two plane axes at $n_2$, read on the L13 residual cloud', fontsize=TITLE_FS)
plt.tight_layout(); savefig(fig, 'w2_projections'); plt.show()

for k in (1, 2):
    print(f'w{k}: |corr| with ' + '  '.join(
        f'log y{j} {abs(np.corrcoef(PROJ[k], np.log(YV[:, j - 1]))[0, 1]):.3f}'
        for j in (1, 2, 3)))


# 11. Killing the 12 neurons and re-measuring the task

Zero-ablation, at **every** position rather than at the patch site: the 12 neurons'
post-SwiGLU activations are set to 0, so their write leaves the residual entirely. Accuracy is
exact-match on the greedily decoded answer, over operand lists of length
$k \in \{2, 3, 4, 5, 10, 20\}$. Two random sets of 12 are the control.

In [ ]:
# ============================================================================================
# SELF-CONTAINED.  Needs nothing from the cells above -- not `cached`, not `PATH`, not `SEL`.
# Safe to run on its own in a live kernel or a fresh one; it loads the model only if needed.
# ============================================================================================
import os, importlib
import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt

import data_numberreps as D
import data_comparator as C
import data_multicompare as MC
import data_multicompare2 as M2
importlib.reload(M2)
from data_numberreps import device

A_RESULTS = 'results'
A_REGIME = '2digit'
A_PATH = f'{A_RESULTS}/multicompare2_ablate_{A_REGIME}.npz'
A_FORCE = False                    # True to recompute even though the file exists

A_KS = [2, 3, 4, 5, 10, 20]        # number of operands in the list
A_N = 200                          # prompts per length
A_SEEDS = [0, 1]                   # random-set controls
A_SEED = 0
A_LOW, A_HIGH = (MC.REGIMES[A_REGIME][k] for k in ('low', 'high'))
A_DIGITS = len(str(A_HIGH - 1))    # operand (and answer) digit count
A_LAYERS, A_DFF = [14, 15], 18944
_PL = np.concatenate([np.full(A_DFF, l) for l in A_LAYERS])
_PW = np.concatenate([np.arange(A_DFF) for _ in A_LAYERS])
A_NPOOL = len(_PL)
A_NAME = lambda p: f'L{_PL[p]}#{_PW[p]}'

# The neuron set, read straight from the two-number results rather than from `SEL`.
A_SEL = np.asarray(np.load(f'{A_RESULTS}/comparator_data_{A_REGIME}.npz')['overlap_k20'])


def a_handles(pooled):
    """Zero those pooled neurons in MLP14 / MLP15 at every position."""
    pooled = np.asarray(pooled)
    sel = {l: torch.as_tensor(_PW[pooled[_PL[pooled] == l]], device=device, dtype=torch.long)
           for l in A_LAYERS}
    return lambda: [C.mlp_of(l).down_proj.register_forward_pre_hook(M2.make_neuron_zero_hook(ii))
                    for l, ii in sel.items() if len(ii)]


def _ablate():
    if getattr(D, 'model', None) is None:
        D.load_model()
    out = {'ks': np.array(A_KS), 'n': np.array(A_N), 'digits': np.array(A_DIGITS),
           'sel': A_SEL}
    rng = np.random.default_rng(31)
    sets = {'killed': A_SEL}
    for s in A_SEEDS:
        sets[f'rand{s}'] = rng.choice(A_NPOOL, len(A_SEL), replace=False)
    for nm, pooled in sets.items():
        out[f'pooled_{nm}'] = np.asarray(pooled)
    for k in A_KS:
        nums = M2.sample_max_prompts(k, A_N, A_SEED + 500 + k, A_LOW, A_HIGH)
        out[f'intact_k{k}'] = M2.task_accuracy(nums, A_DIGITS)
        for nm, pooled in sets.items():
            out[f'{nm}_k{k}'] = M2.task_accuracy(nums, A_DIGITS, a_handles(pooled))
        print(f'  k={k:2d}: intact {out[f"intact_k{k}"].mean():.3f}   '
              f'{len(A_SEL)} killed {out[f"killed_k{k}"].mean():.3f}   '
              f'random {np.mean([out[f"rand{s}_k{k}"].mean() for s in A_SEEDS]):.3f}',
              flush=True)
    return out


if os.path.exists(A_PATH) and not A_FORCE:
    ABL = dict(np.load(A_PATH))
    print(f'loaded {A_PATH}   ({len(ABL)} arrays)')
else:
    ABL = _ablate()
    np.savez_compressed(A_PATH, **{k: np.asarray(v) for k, v in ABL.items()})
    print(f'computed and saved {A_PATH}   ({len(ABL)} arrays)')

AKS = ABL['ks'].tolist()
A_LFS, A_TFS, A_GFS = 16, 16, 12
A_CLR = {'intact': '#6E6E6E', 'killed': '#2166AC', 'rand': '#C06A9B'}


def acc(nm, k):
    v = ABL[f'{nm}_k{k}'].astype(float)
    return v.mean(), v.std(ddof=1) / np.sqrt(len(v))


def a_style(ax, xlabel=None, ylabel=None, title=None):
    if xlabel: ax.set_xlabel(xlabel, fontsize=A_LFS)
    if ylabel: ax.set_ylabel(ylabel, fontsize=A_LFS)
    if title:  ax.set_title(title, fontsize=A_TFS)
    ax.tick_params(labelsize=A_GFS); ax.grid(alpha=0.3)
    for s_ in ('top', 'right'):
        ax.spines[s_].set_visible(False)


# --- the three-number case on its own -----------------------------------------------------
K0 = 3
bars = [('intact', 'intact model', A_CLR['intact']),
        ('killed', f'the {len(A_SEL)} neurons zeroed', A_CLR['killed'])]
bars += [(f'rand{s}', f'{len(A_SEL)} random zeroed' if s == A_SEEDS[0] else '_nolegend_',
          A_CLR['rand']) for s in A_SEEDS]

fig, ax = plt.subplots(figsize=(7.8, 5.0))
for bi, (nm, lab, col) in enumerate(bars):
    m, se = acc(nm, K0)
    ax.bar(bi, m, 0.68, yerr=se, capsize=4, color=col, alpha=0.92, label=lab)
    ax.text(bi, m + se + 0.02, f'{m:.3f}', ha='center', va='bottom', fontsize=11)
ax.set_xticks(range(len(bars)))
ax.set_xticklabels(['intact', f'the {len(A_SEL)}', 'rand a', 'rand b'], fontsize=A_LFS)
ax.set_ylim(0, 1.12)
a_style(ax, ylabel=f'accuracy on max of {K0} numbers',
        title=f'zero-ablating the two-number neurons  ($k={K0}$, {int(ABL["n"])} prompts)')
ax.legend(fontsize=A_GFS)
fig.savefig('figs/multicompare2_ablate_k3.png', dpi=300, bbox_inches='tight')
plt.show()

m0, m1 = acc('intact', K0)[0], acc('killed', K0)[0]
mr = np.mean([acc(f'rand{s}', K0)[0] for s in A_SEEDS])
print(f'k={K0}: intact {m0:.3f} -> killed {m1:.3f}  '
      f'(drop {m0 - m1:+.3f}, {100 * (m0 - m1) / max(m0, 1e-9):.1f}% relative)   '
      f'random control {mr:.3f}')
print('neurons zeroed: ' + ', '.join(A_NAME(p) for p in A_SEL))

In [ ]:
# Self-contained given the cell above (uses only ABL, AKS, acc, A_* from it).
SER = [('intact', 'intact model', A_CLR['intact'], '-', 'o'),
       ('killed', f'the {len(A_SEL)} two-number neurons zeroed', A_CLR['killed'], '-', 's')]
SER += [(f'rand{s}', f'{len(A_SEL)} random zeroed' if s == A_SEEDS[0] else '_nolegend_',
         A_CLR['rand'], '--', '^') for s in A_SEEDS]

fig, axes = plt.subplots(1, 2, figsize=(15, 5.2), width_ratios=[1.15, 1])

ax = axes[0]
for nm, lab, col, ls, mk in SER:
    m = np.array([acc(nm, k)[0] for k in AKS])
    e = np.array([acc(nm, k)[1] for k in AKS])
    ax.errorbar(AKS, m, yerr=e, fmt=mk, ls=ls, ms=6, color=col, lw=1.9, capsize=3,
                alpha=0.95 if 'rand' not in nm else 0.75, label=lab)
ax.set_xscale('log'); ax.set_xticks(AKS)
ax.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
ax.set_ylim(0, 1.05)
a_style(ax, xlabel='numbers in the list, $k$', ylabel='accuracy',
        title='task accuracy vs list length')
ax.legend(fontsize=A_GFS)

ax = axes[1]
base = np.array([acc('intact', k)[0] for k in AKS])
for nm, lab, col, ls, mk in SER[1:]:
    ax.plot(AKS, base - np.array([acc(nm, k)[0] for k in AKS]), mk, ls=ls, ms=6, color=col,
            lw=1.9, alpha=0.95 if 'rand' not in nm else 0.75, label=lab)
ax.axhline(0, color='0.6', lw=0.9)
ax.set_xscale('log'); ax.set_xticks(AKS)
ax.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
a_style(ax, xlabel='numbers in the list, $k$', ylabel='accuracy lost vs the intact model',
        title='size of the ablation effect')
ax.legend(fontsize=A_GFS)

fig.suptitle(f'zero-ablating {len(A_SEL)} MLP14/15 neurons, {int(ABL["n"])} prompts per length',
             fontsize=18)
plt.tight_layout()
fig.savefig('figs/multicompare2_ablate_vs_length.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'{"k":>4}{"intact":>10}{"killed":>10}{"drop":>9}{"random":>10}')
for k in AKS:
    mr = np.mean([acc(f'rand{s}', k)[0] for s in A_SEEDS])
    print(f'{k:>4}{acc("intact", k)[0]:>10.3f}{acc("killed", k)[0]:>10.3f}'
          f'{acc("intact", k)[0] - acc("killed", k)[0]:>9.3f}{mr:>10.3f}')

### Only at the last number position

The same 12 neurons zeroed at a **single** token — the last token of the final operand, the
$k$-operand generalisation of $n_3$ — instead of everywhere. The gap between the two curves
separates "needed somewhere in the forward pass" from "needed at the token where the comparison
is read".

In [ ]:
# Self-contained given the first cell of this section (uses ABL, acc, A_* from it).
import importlib
importlib.reload(M2)

A_PATH_LP = f'{A_RESULTS}/multicompare2_ablate_lastpos_{A_REGIME}.npz'
A_FORCE_LP = False


def a_handles_at(pooled, pos):
    """Zero those pooled neurons in MLP14 / MLP15 at one position only."""
    pooled = np.asarray(pooled)
    sel = {l: torch.as_tensor(_PW[pooled[_PL[pooled] == l]], device=device, dtype=torch.long)
           for l in A_LAYERS}
    return lambda: [C.mlp_of(l).down_proj.register_forward_pre_hook(
        M2.make_neuron_zero_hook_at(ii, pos)) for l, ii in sel.items() if len(ii)]


def _ablate_lastpos():
    if getattr(D, 'model', None) is None:
        D.load_model()
    out = {'ks': np.array(A_KS)}
    rng = np.random.default_rng(31)          # same draws as the all-positions cell
    sets = {'killed': A_SEL}
    for s in A_SEEDS:
        sets[f'rand{s}'] = rng.choice(A_NPOOL, len(A_SEL), replace=False)
    for k in A_KS:
        nums = M2.sample_max_prompts(k, A_N, A_SEED + 500 + k, A_LOW, A_HIGH)
        pos = M2.last_number_pos(nums[0])
        assert all(M2.last_number_pos(n) == pos for n in nums[:8]), \
            'the read-out position must be the same for every prompt at this k'
        out[f'pos_k{k}'] = np.array(pos)
        for nm, pooled in sets.items():
            out[f'{nm}_k{k}'] = M2.task_accuracy(nums, A_DIGITS, a_handles_at(pooled, pos))
        print(f'  k={k:2d} @tok{pos}: {len(A_SEL)} killed {out[f"killed_k{k}"].mean():.3f}   '
              f'random {np.mean([out[f"rand{s}_k{k}"].mean() for s in A_SEEDS]):.3f}   '
              f'(everywhere {ABL[f"killed_k{k}"].mean():.3f}, '
              f'intact {ABL[f"intact_k{k}"].mean():.3f})', flush=True)
    return out


if os.path.exists(A_PATH_LP) and not A_FORCE_LP:
    ABLP = dict(np.load(A_PATH_LP))
    print(f'loaded {A_PATH_LP}   ({len(ABLP)} arrays)')
else:
    ABLP = _ablate_lastpos()
    np.savez_compressed(A_PATH_LP, **{k: np.asarray(v) for k, v in ABLP.items()})
    print(f'computed and saved {A_PATH_LP}   ({len(ABLP)} arrays)')


def accp(nm, k):
    v = ABLP[f'{nm}_k{k}'].astype(float)
    return v.mean(), v.std(ddof=1) / np.sqrt(len(v))


SER_LP = [('intact model', acc, 'intact', A_CLR['intact'], '-', 'o'),
          (f'the {len(A_SEL)} zeroed everywhere', acc, 'killed', A_CLR['killed'], '-', 's'),
          (f'the {len(A_SEL)} zeroed at the last number', accp, 'killed', '#C97B00', '-', 'D'),
          (f'{len(A_SEL)} random at the last number', accp, 'rand0', A_CLR['rand'], '--', '^')]

fig, axes = plt.subplots(1, 2, figsize=(15, 5.2), width_ratios=[1.15, 1])

ax = axes[0]
for lab, fn, key, col, ls, mk in SER_LP:
    m = np.array([fn(key, k)[0] for k in AKS])
    e = np.array([fn(key, k)[1] for k in AKS])
    ax.errorbar(AKS, m, yerr=e, fmt=mk, ls=ls, ms=6, color=col, lw=1.9, capsize=3,
                alpha=0.95 if 'random' not in lab else 0.75, label=lab)
ax.set_xscale('log'); ax.set_xticks(AKS)
ax.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
ax.set_ylim(0, 1.05)
a_style(ax, xlabel='numbers in the list, $k$', ylabel='accuracy',
        title='everywhere vs at the read-out token')
ax.legend(fontsize=A_GFS - 1)

ax = axes[1]
base = np.array([acc('intact', k)[0] for k in AKS])
for lab, fn, key, col, ls, mk in SER_LP[1:]:
    ax.plot(AKS, base - np.array([fn(key, k)[0] for k in AKS]), mk, ls=ls, ms=6, color=col,
            lw=1.9, alpha=0.95 if 'random' not in lab else 0.75, label=lab)
ax.axhline(0, color='0.6', lw=0.9)
ax.set_xscale('log'); ax.set_xticks(AKS)
ax.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
a_style(ax, xlabel='numbers in the list, $k$', ylabel='accuracy lost vs the intact model',
        title='how much of the effect is at that one token')
ax.legend(fontsize=A_GFS - 1)

fig.suptitle('zero-ablation: all positions vs the last number position only', fontsize=18)
plt.tight_layout()
fig.savefig('figs/multicompare2_ablate_lastpos.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'{"k":>4}{"tok":>6}{"intact":>10}{"everywhere":>12}{"last-num":>11}{"rand@last":>11}'
      f'{"frac at tok":>13}')
for k in AKS:
    i_, e_, l_ = acc('intact', k)[0], acc('killed', k)[0], accp('killed', k)[0]
    r_ = accp('rand0', k)[0]
    fr = (i_ - l_) / (i_ - e_) if abs(i_ - e_) > 1e-9 else float('nan')
    print(f'{k:>4}{int(ABLP[f"pos_k{k}"]):>6}{i_:>10.3f}{e_:>12.3f}{l_:>11.3f}{r_:>11.3f}'
          f'{fr:>13.2f}')

### Split by whether the last number is the maximum

When the final operand *is* the maximum, the answer is sitting at the read-out token already;
otherwise it has to be carried there from an earlier position. No new forward passes — the
operand lists are regenerated from the same seeds on CPU and used to index the stored
per-prompt correctness arrays.

In [ ]:
# Self-contained given the first cell of this section. Pure re-analysis: `sample_max_prompts` is
# deterministic and CPU-only, so the split costs nothing and the model is never touched.
A_MASK = {}
for k in A_KS:
    nums = M2.sample_max_prompts(k, A_N, A_SEED + 500 + k, A_LOW, A_HIGH)
    A_MASK[k] = np.array([int(np.argmax(t)) == k - 1 for t in nums], dtype=bool)
print('P(last operand is the max), expected 1/k:')
print('  ' + '   '.join(f'k={k}: {A_MASK[k].mean():.3f} ({A_MASK[k].sum()}/{A_N})'
                        for k in A_KS))

HAS_LP = 'ABLP' in globals()
A_SPLIT = [('intact', lambda nm, k: ABL[f'{nm}_k{k}'], 'intact', 'intact model',
            A_CLR['intact'], 'o'),
           ('killed', lambda nm, k: ABL[f'{nm}_k{k}'], 'killed',
            f'the {len(A_SEL)} zeroed everywhere', A_CLR['killed'], 's')]
if HAS_LP:
    A_SPLIT.append(('killed_lp', lambda nm, k: ABLP[f'killed_k{k}'], 'killed',
                    f'the {len(A_SEL)} zeroed at the last number', '#C97B00', 'D'))


def sub(getter, nm, k, want_last):
    """Mean and SE of one condition's correctness over one subgroup."""
    v = np.asarray(getter(nm, k)).astype(float)
    m = A_MASK[k] if want_last else ~A_MASK[k]
    v = v[m]
    if len(v) < 2:
        return np.nan, np.nan, len(v)
    return v.mean(), v.std(ddof=1) / np.sqrt(len(v)), len(v)


fig, axes = plt.subplots(1, 2, figsize=(15, 5.2), sharey=True)
for ax, want_last, ttl in zip(axes, (True, False),
                              ('the last number IS the max',
                               'the max is an earlier number')):
    for key, getter, nm, lab, col, mk in A_SPLIT:
        pts = [sub(getter, nm, k, want_last) for k in AKS]
        m = np.array([p[0] for p in pts]); e = np.array([p[1] for p in pts])
        ax.errorbar(AKS, m, yerr=e, fmt=mk, ls='-', ms=6, color=col, lw=1.9, capsize=3,
                    alpha=0.95, label=lab)
    ax.set_xscale('log'); ax.set_xticks(AKS)
    ax.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
    ax.set_ylim(0, 1.05)
    a_style(ax, xlabel='numbers in the list, $k$',
            ylabel='accuracy' if want_last else None, title=ttl)
    if want_last:
        ax.legend(fontsize=A_GFS - 1)
    # n per point, so the thin k=20 subgroup is visible rather than implied
    for k, p in zip(AKS, [sub(A_SPLIT[0][1], 'intact', k, want_last) for k in AKS]):
        ax.annotate(f'n={p[2]}', (k, 0.03), ha='center', fontsize=8, color='0.35')

fig.suptitle('accuracy split by whether the maximum sits at the read-out token', fontsize=18)
plt.tight_layout()
fig.savefig('figs/multicompare2_ablate_by_maxpos.png', dpi=300, bbox_inches='tight')
plt.show()

hdr = f'{"k":>4}{"subset":>14}{"n":>5}' + ''.join(f'{l[:16]:>18}' for _, _, _, l, _, _ in A_SPLIT)
print(hdr)
for k in AKS:
    for want_last, tag in ((True, 'last is max'), (False, 'max earlier')):
        row = f'{k:>4}{tag:>14}'
        n = sub(A_SPLIT[0][1], 'intact', k, want_last)[2]
        row += f'{n:>5}'
        for key, getter, nm, lab, col, mk in A_SPLIT:
            row += f'{sub(getter, nm, k, want_last)[0]:>18.3f}'
        print(row)